# Model Training

This notebook is optimized for the final deadline sprint.

It does four things:

1. builds a strict grouped evaluation pipeline using the existing `Region` column,
2. tests a small shortlist of target-specific candidates,
3. freezes a safe manifest from full grouped CV,
4. writes three submission files:
   - **A** = safe anchor,
   - **B** = EC aggressive + DRP safe,
   - **C** = hedge blend.

Notes:
- We assume `Region` already exists in the provided dataset.
- We keep the notebook cell-by-cell and avoid one giant integrated script.
- We clip predictions to nonnegative values before submission.

In [1]:
import os
import sys
import json
import time
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans

from xgboost import XGBRegressor
from IPython.display import display

## Environment and MLflow

In [2]:
sys.path.append(os.path.abspath('..'))

ENV = 'local'   # switch to 'snowflake' if needed

if ENV == 'local':
    from src import config_local as config
else:
    from src import config_snowflake as config

mlflow.set_tracking_uri(config.MLFLOW_URI)
mlflow.set_experiment('WaterQuality')

print('MLflow URI:', config.MLFLOW_URI)

2026/03/12 22:00:44 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/03/12 22:00:44 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/03/12 22:00:44 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/03/12 22:00:44 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/03/12 22:00:44 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/03/12 22:00:44 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/03/12 22:00:44 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/12 22:00:44 INFO alembic.runtime.migration: Will assume non-transactional DDL.


MLflow URI: sqlite:///../mlflow.db


## Global config

This cell defines:
- targets,
- split metadata,
- artifact directory,
- hashing helpers,
- submission integrity checks.

In [3]:
TARGET_COLS = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus'
]

SPLIT_STRATEGY = 'SpatialGroupKFold+PseudoHoldoutGroups'
GROUP_DEFINITION_VERSION = 'kmeans_latlon_v2_group_holdout'
PIPELINE_VERSION = 'deadline_v2_mvp4_spatial_group_holdout'
PREPROCESS_VERSION = 'median_scaler'
ARTIFACT_DIR = '../models/final_deadline_mvp4'

SPATIAL_N_CLUSTERS = 16
CV_N_SPLITS = 5
HOLDOUT_MARGIN_DEG = 0.35
HOLDOUT_MIN_GROUPS = 3
HOLDOUT_MIN_FRAC = 0.08
HOLDOUT_MAX_FRAC = 0.15

os.makedirs(ARTIFACT_DIR, exist_ok=True)


def hash_str(s: str) -> str:
    '''
    Create a short stable hash from a string.
    '''
    return hashlib.sha256(s.encode('utf-8')).hexdigest()[:16]


def hash_list(values) -> str:
    '''
    Hash a list of values after converting to strings.
    '''
    return hash_str('||'.join(map(str, values)))


def compute_group_values_hash(groups: pd.Series) -> str:
    '''
    Hash the exact ordered group assignments.
    Useful to ensure runs are truly comparable.
    '''
    return hash_list(groups.fillna('NA').astype(str).tolist())


def compute_feature_set_hash(features: list) -> str:
    '''
    Hash a feature list in sorted form.
    '''
    return hash_list(sorted(features))


def target_key(target_name: str) -> str:
    '''
    Make a target name filename-safe.
    '''
    return target_name.replace(' ', '')


def make_row_id_template(template_df: pd.DataFrame) -> pd.DataFrame:
    '''
    Add an immutable row_id to the submission template
    so row order can be validated before saving.
    '''
    out = template_df.copy()
    out['row_id'] = np.arange(len(out), dtype=int)
    return out


def assert_submission_integrity(sub_df: pd.DataFrame, template_df: pd.DataFrame, target_cols: list):
    '''
    Validate that the submission is structurally safe.
    '''
    if len(sub_df) != len(template_df):
        raise RuntimeError(f'Row count mismatch: sub={len(sub_df)} template={len(template_df)}')

    if 'row_id' not in sub_df.columns or 'row_id' not in template_df.columns:
        raise RuntimeError('row_id missing in submission/template.')

    if sub_df['row_id'].duplicated().any():
        raise RuntimeError('Duplicate row_id in submission.')

    if not sub_df['row_id'].equals(template_df['row_id']):
        raise RuntimeError('row_id order mismatch.')

    if sub_df[target_cols].isnull().any().any():
        raise RuntimeError('NaN found in target predictions.')

    if (sub_df[target_cols] < 0).any().any():
        raise RuntimeError('Negative predictions found.')


print('Global config loaded.')

Global config loaded.


## Data loading

We load the training data and confirm that the `Region` column already exists.

In [4]:
TRAIN_PATH = '../data/interim/water_quality_mvp_baseline.parquet'
VALID_PATH = '../data/interim/water_quality_mvp_validation.parquet'


df = pd.read_parquet(TRAIN_PATH).copy()
df_val_geo = pd.read_parquet(VALID_PATH)[['Latitude', 'Longitude', 'Sample Date']].copy()

df['Sample Date'] = pd.to_datetime(df['Sample Date'], errors='coerce')
df_val_geo['Sample Date'] = pd.to_datetime(df_val_geo['Sample Date'], errors='coerce')

required_cols = ['Latitude', 'Longitude', 'Sample Date'] + TARGET_COLS
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise RuntimeError(f'Missing required training columns: {missing_cols}')


def add_spatial_groups(data: pd.DataFrame, n_clusters: int = SPATIAL_N_CLUSTERS) -> pd.DataFrame:
    '''
    Create stable spatial groups from latitude/longitude using KMeans.
    '''
    out = data.copy()
    n_clusters = min(max(4, int(n_clusters)), len(out))

    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
    out['spatial_group'] = km.fit_predict(out[['Latitude', 'Longitude']].astype(float)).astype(str)
    return out


def select_pseudo_holdout_groups(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    min_groups: int = HOLDOUT_MIN_GROUPS,
    min_frac: float = HOLDOUT_MIN_FRAC,
    max_frac: float = HOLDOUT_MAX_FRAC,
    margin_deg: float = HOLDOUT_MARGIN_DEG,
):
    '''
    Select whole spatial groups nearest to the validation footprint.
    The selection targets a row fraction range and enforces a minimum number of groups.
    '''
    gdf = train_df.groupby('spatial_group', as_index=False).agg(
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean'),
        n=('spatial_group', 'size')
    )

    lat_min = float(valid_df['Latitude'].min()) - margin_deg
    lat_max = float(valid_df['Latitude'].max()) + margin_deg
    lon_min = float(valid_df['Longitude'].min()) - margin_deg
    lon_max = float(valid_df['Longitude'].max()) + margin_deg

    valid_center_lat = float(valid_df['Latitude'].mean())
    valid_center_lon = float(valid_df['Longitude'].mean())

    lat = gdf['Latitude'].astype(float)
    lon = gdf['Longitude'].astype(float)

    lat_gap = np.maximum(np.maximum(lat_min - lat, 0.0), lat - lat_max)
    lon_gap = np.maximum(np.maximum(lon_min - lon, 0.0), lon - lon_max)

    gdf['bbox_dist'] = np.sqrt(lat_gap ** 2 + lon_gap ** 2)
    gdf['center_dist'] = np.sqrt((lat - valid_center_lat) ** 2 + (lon - valid_center_lon) ** 2)

    gdf = gdf.sort_values(['bbox_dist', 'center_dist', 'n'], ascending=[True, True, False]).reset_index(drop=True)

    total_rows = int(len(train_df))
    selected = []
    selected_rows = 0

    for _, row in gdf.iterrows():
        group_name = str(row['spatial_group'])
        group_rows = int(row['n'])

        need_groups = len(selected) < int(min_groups)
        need_rows = (selected_rows / total_rows) < float(min_frac)

        if need_groups or need_rows:
            selected.append(group_name)
            selected_rows += group_rows
            continue

        next_frac = (selected_rows + group_rows) / total_rows
        if next_frac <= float(max_frac):
            selected.append(group_name)
            selected_rows += group_rows
        else:
            break

    i = len(selected)
    while (len(selected) < int(min_groups) or (selected_rows / total_rows) < float(min_frac)) and i < len(gdf):
        group_name = str(gdf.loc[i, 'spatial_group'])
        if group_name not in selected:
            selected.append(group_name)
            selected_rows += int(gdf.loc[i, 'n'])
        i += 1

    all_groups = set(train_df['spatial_group'].astype(str).unique().tolist())
    if len(selected) >= len(all_groups):
        selected = selected[:-1]

    selected = sorted(set(selected), key=lambda x: int(x))

    return selected


df = add_spatial_groups(df, n_clusters=SPATIAL_N_CLUSTERS)
selected_holdout_groups = select_pseudo_holdout_groups(
    df,
    df_val_geo,
    min_groups=HOLDOUT_MIN_GROUPS,
    min_frac=HOLDOUT_MIN_FRAC,
    max_frac=HOLDOUT_MAX_FRAC,
    margin_deg=HOLDOUT_MARGIN_DEG,
)

df['is_pseudo_valid'] = df['spatial_group'].astype(str).isin(selected_holdout_groups)

print('Training shape:', df.shape)
print('Spatial groups:', df['spatial_group'].nunique())
print('Selected holdout groups:', selected_holdout_groups)
print('Pseudo-validation rows:', int(df['is_pseudo_valid'].sum()))
print('Pseudo-validation share:', round(float(df['is_pseudo_valid'].mean()), 4))
print('\nPseudo-validation by group:')
print(df.groupby('spatial_group')['is_pseudo_valid'].sum().sort_values(ascending=False).head(16))

Training shape: (9319, 12)
Spatial groups: 16
Selected holdout groups: ['3', '9', '10']
Pseudo-validation rows: 1433
Pseudo-validation share: 0.1538

Pseudo-validation by group:
spatial_group
3     509
10    477
9     447
0       0
12      0
13      0
1       0
11      0
15      0
14      0
4       0
2       0
5       0
6       0
7       0
8       0
Name: is_pseudo_valid, dtype: int64


In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9319 entries, 0 to 9318
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Latitude                       9319 non-null   float64       
 1   Longitude                      9319 non-null   float64       
 2   Sample Date                    9319 non-null   datetime64[ns]
 3   swir22                         8234 non-null   float64       
 4   NDMI                           8234 non-null   float64       
 5   MNDWI                          8234 non-null   float64       
 6   pet                            9319 non-null   float64       
 7   Total Alkalinity               9319 non-null   float64       
 8   Electrical Conductance         9319 non-null   float64       
 9   Dissolved Reactive Phosphorus  9319 non-null   float64       
 10  spatial_group                  9319 non-null   object        
 11  is_pseudo_valid  

## Feature engineering

This notebook only uses the two engineered features that were explicitly confirmed from the winning setup:

- `pop_density_upstream`
- `specific_discharge`

In [ ]:
def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    '''
    Keep this as a pass-through for MVP 4-feature experiments.
    '''
    return data.copy()


df = engineer_features(df)
print('Feature engineering step completed (no-op for MVP 4-feature setup).')

## Frozen feature sets

We use two frozen feature sets:

- **A** = stable base set
- **B** = base + empirical interactions

In [6]:
# Benchmark-like local signal set only
BENCHMARK_4 = ['swir22', 'NDMI', 'MNDWI', 'pet']

FEATURE_SETS = {
    'C': BENCHMARK_4,
}


def features_for_set(df_local: pd.DataFrame, fs_name: str):
    '''
    Return a strict feature list for a named feature set.
    Fail fast if expected columns are missing.
    '''
    requested = FEATURE_SETS[fs_name]
    missing = [f for f in requested if f not in df_local.columns]
    if missing:
        raise RuntimeError(f'Missing in {fs_name}: {missing}')
    return requested


print('Feature sets ready:')
for k, v in FEATURE_SETS.items():
    print(f'{k}: {len(v)} features -> {v}')

Feature sets ready:
C: 4 features -> ['swir22', 'NDMI', 'MNDWI', 'pet']


## Preprocessing

We use median imputation and standard scaling inside the CV pipeline.

In [7]:
def get_preprocessor(features_used):
    '''
    Build the preprocessing pipeline for numeric features.
    '''
    numeric_pipe = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    return ColumnTransformer(
        transformers=[('num', numeric_pipe, features_used)],
        remainder='drop'
    )

## Models and shortlist

This shortlist is deliberately small:
- TA: mostly stable XGB
- EC: linear empirical + one XGB challenger
- DRP: safer linear options + one shallow XGB challenger

In [8]:
from sklearn.ensemble import RandomForestRegressor


def log_wrap(model):
    # Apply log transform on targets for optional RF log-variant.
    return TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )


# RF-only defaults for this iteration
DEFAULT_RF_PARAMS = {
    'n_estimators': 600,
    'min_samples_leaf': 3,
    'max_features': 'sqrt',
    'random_state': 42,
    'n_jobs': -1,
}

MODEL_SPECS = {
    'RF_n600_raw': {'kind': 'rf', 'params': {}, 'log_target': False},
    'RF_n600_Log': {'kind': 'rf', 'params': {}, 'log_target': True},
}


def build_model_from_spec(spec):
    kind = spec['kind']
    params = spec.get('params', {})
    use_log = spec.get('log_target', True)

    if kind == 'rf':
        p = DEFAULT_RF_PARAMS.copy()
        p.update(params)
        base = RandomForestRegressor(**p)
    else:
        raise ValueError(f'Unknown model kind for RF-only mode: {kind}')

    return log_wrap(base) if use_log else base


MODEL_BANK = {name: build_model_from_spec(spec) for name, spec in MODEL_SPECS.items()}

TARGET_SWEEP = {
    'Total Alkalinity': [
        ('C', 'RF_n600_raw'),
        ('C', 'RF_n600_Log'),
    ],
    'Electrical Conductance': [
        ('C', 'RF_n600_raw'),
        ('C', 'RF_n600_Log'),
    ],
    'Dissolved Reactive Phosphorus': [
        ('C', 'RF_n600_raw'),
        ('C', 'RF_n600_Log'),
    ],
}

print('RF-only sweep active for all targets.')
display(pd.DataFrame(
    [(t, fs, m) for t, recipes in TARGET_SWEEP.items() for fs, m in recipes],
    columns=['target', 'feature_set', 'model']
))



RF-only sweep active for all targets.


,target,feature_set,model
0,Total Alkalinity,C,RF_n600_raw
1,Total Alkalinity,C,RF_n600_Log
2,Electrical Conductance,C,RF_n600_raw
3,Electrical Conductance,C,RF_n600_Log
4,Dissolved Reactive Phosphorus,C,RF_n600_raw
5,Dissolved Reactive Phosphorus,C,RF_n600_Log


## Grouped evaluation helpers

This cell:
- runs grouped out-of-fold predictions,
- reports worst-region behavior,
- saves final artifacts for finalists.

In [9]:
def grouped_oof_eval(df_local, target, estimator, features_used, allowed_regions=None):
    # Run grouped OOF evaluation with GroupKFold on spatial clusters,
    # plus a dummy median baseline on exactly the same folds/holdout.
    d = df_local.copy()

    if allowed_regions is not None:
        allowed = set(pd.Series(allowed_regions).astype(str).tolist())
        d = d[d['spatial_group'].astype(str).isin(allowed)].copy()

    X = d[features_used].reset_index(drop=True)
    y = d[target].astype(float).reset_index(drop=True)
    groups = d['spatial_group'].astype(str).reset_index(drop=True)

    n_groups = int(groups.nunique())
    if n_groups < 2:
        raise RuntimeError('Need at least 2 spatial groups for grouped CV.')

    n_splits = min(int(CV_N_SPLITS), n_groups)
    gkf = GroupKFold(n_splits=n_splits)

    pred = np.full(len(d), np.nan, dtype=float)
    dummy_pred = np.full(len(d), np.nan, dtype=float)

    fold_rows = []
    dummy_fold_rows = []

    for fold_id, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
        pipe = Pipeline([
            ('preprocessor', get_preprocessor(features_used)),
            ('model', clone(estimator))
        ])

        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        pipe.fit(X_tr, y_tr)
        fold_pred = np.asarray(pipe.predict(X_te), dtype=float)
        pred[test_idx] = fold_pred

        fold_rows.append({
            'fold': fold_id,
            'n': int(len(test_idx)),
            'r2': float(r2_score(y_te, fold_pred)),
            'rmse': float(np.sqrt(mean_squared_error(y_te, fold_pred))),
            'mae': float(mean_absolute_error(y_te, fold_pred)),
        })

        dummy_value = float(np.median(y_tr))
        dummy_fold_pred = np.full(len(test_idx), dummy_value, dtype=float)
        dummy_pred[test_idx] = dummy_fold_pred

        dummy_fold_rows.append({
            'fold': fold_id,
            'n': int(len(test_idx)),
            'r2': float(r2_score(y_te, dummy_fold_pred)),
            'rmse': float(np.sqrt(mean_squared_error(y_te, dummy_fold_pred))),
            'mae': float(mean_absolute_error(y_te, dummy_fold_pred)),
        })

    if np.isnan(pred).any():
        raise RuntimeError('OOF predictions contain NaN values.')
    if np.isnan(dummy_pred).any():
        raise RuntimeError('Dummy OOF predictions contain NaN values.')

    fold_df = pd.DataFrame(fold_rows)
    dummy_fold_df = pd.DataFrame(dummy_fold_rows)

    holdout_r2 = np.nan
    dummy_holdout_r2 = np.nan

    if 'is_pseudo_valid' in d.columns:
        hold_mask = d['is_pseudo_valid'].astype(bool).reset_index(drop=True)

        if hold_mask.any() and int((~hold_mask).sum()) >= 2 and int(hold_mask.sum()) >= 2:
            hold_train_groups = set(groups.loc[~hold_mask].tolist())
            hold_test_groups = set(groups.loc[hold_mask].tolist())

            if hold_train_groups.intersection(hold_test_groups):
                raise RuntimeError('Pseudo-holdout leakage detected: train and holdout share groups.')

            hold_pipe = Pipeline([
                ('preprocessor', get_preprocessor(features_used)),
                ('model', clone(estimator))
            ])

            hold_pipe.fit(X.loc[~hold_mask], y.loc[~hold_mask])
            hold_pred = np.asarray(hold_pipe.predict(X.loc[hold_mask]), dtype=float)
            holdout_r2 = float(r2_score(y.loc[hold_mask], hold_pred))

            dummy_hold_value = float(np.median(y.loc[~hold_mask]))
            dummy_hold_pred = np.full(int(hold_mask.sum()), dummy_hold_value, dtype=float)
            dummy_holdout_r2 = float(r2_score(y.loc[hold_mask], dummy_hold_pred))

    model_r2 = float(r2_score(y, pred))
    model_rmse = float(np.sqrt(mean_squared_error(y, pred)))
    model_mae = float(mean_absolute_error(y, pred))
    model_mean_fold_r2 = float(fold_df['r2'].mean()) if not fold_df.empty else np.nan
    model_min_fold_r2 = float(fold_df['r2'].min()) if not fold_df.empty else np.nan

    dummy_r2 = float(r2_score(y, dummy_pred))
    dummy_rmse = float(np.sqrt(mean_squared_error(y, dummy_pred)))
    dummy_mae = float(mean_absolute_error(y, dummy_pred))
    dummy_mean_fold_r2 = float(dummy_fold_df['r2'].mean()) if not dummy_fold_df.empty else np.nan
    dummy_min_fold_r2 = float(dummy_fold_df['r2'].min()) if not dummy_fold_df.empty else np.nan

    delta_holdout = np.nan
    if not pd.isna(holdout_r2) and not pd.isna(dummy_holdout_r2):
        delta_holdout = float(holdout_r2 - dummy_holdout_r2)

    return {
        'pred': pred,
        'rmse': model_rmse,
        'mae': model_mae,
        'r2': model_r2,
        'mean_fold_r2': model_mean_fold_r2,
        'min_fold_r2': model_min_fold_r2,
        'holdout_r2': holdout_r2,
        'fold_df': fold_df,
        'n_rows': int(len(d)),
        'n_groups': n_groups,
        'dummy_r2': dummy_r2,
        'dummy_rmse': dummy_rmse,
        'dummy_mae': dummy_mae,
        'dummy_mean_fold_r2': dummy_mean_fold_r2,
        'dummy_min_fold_r2': dummy_min_fold_r2,
        'dummy_holdout_r2': dummy_holdout_r2,
        'delta_r2_vs_dummy': float(model_r2 - dummy_r2),
        'delta_min_fold_r2_vs_dummy': float(model_min_fold_r2 - dummy_min_fold_r2),
        'delta_holdout_r2_vs_dummy': delta_holdout,
    }


GLOBAL_R2_WEIGHT = 0.60
HOLDOUT_R2_WEIGHT = 0.25
MIN_FOLD_R2_WEIGHT = 0.15


def compute_selection_score(overall_r2, holdout_r2, min_fold_r2=None):
    # Build a finalist selection score with holdout awareness and fold robustness.
    holdout_term = 0.0 if pd.isna(holdout_r2) else float(holdout_r2)
    min_fold_term = 0.0 if pd.isna(min_fold_r2) else float(min_fold_r2)

    return float(
        GLOBAL_R2_WEIGHT * float(overall_r2) +
        HOLDOUT_R2_WEIGHT * holdout_term +
        MIN_FOLD_R2_WEIGHT * min_fold_term
    )


def fit_full_and_save(df_local, target, estimator, features_used, run_name):
    # Fit the final full-data pipeline and save preprocessor + model artifacts.
    X = df_local[features_used]
    y = df_local[target].astype(float)

    pre = get_preprocessor(features_used)
    Xp = pre.fit_transform(X)

    mdl = clone(estimator)
    mdl.fit(Xp, y)

    preproc_path = os.path.join(ARTIFACT_DIR, f'{run_name}__preproc.joblib')
    model_path = os.path.join(ARTIFACT_DIR, f'{run_name}__model.joblib')

    joblib.dump(pre, preproc_path)
    joblib.dump(mdl, model_path)

    return preproc_path, model_path



## Stage 1: Scout run

We first run only a narrow shortlist, optionally prioritizing the hardest regions.

In [10]:
SCOUT_GROUPS = None
print('Scout scope: ALL spatial groups')

rows_scout = []

for target in TARGET_COLS:
    for feature_set_name, model_name in TARGET_SWEEP[target]:
        features = features_for_set(df, feature_set_name)
        estimator = clone(MODEL_BANK[model_name])
        run_name = f'SCOUT__{model_name}__{feature_set_name}__{target_key(target)}'

        print()
        print(f'--- {run_name} ---')

        with mlflow.start_run(run_name=run_name):
            t0 = time.time()

            out = grouped_oof_eval(
                df_local=df,
                target=target,
                estimator=estimator,
                features_used=features,
                allowed_regions=SCOUT_GROUPS
            )

            dt = time.time() - t0

            mlflow.log_param('stage', 'scout')
            mlflow.log_param('target', target)
            mlflow.log_param('model_name', model_name)
            mlflow.log_param('feature_set_name', feature_set_name)
            mlflow.log_param('split_strategy', SPLIT_STRATEGY)
            mlflow.log_param('group_definition_version', GROUP_DEFINITION_VERSION)
            mlflow.log_param('group_values_hash', compute_group_values_hash(df['spatial_group'].astype(str)))
            mlflow.log_param('feature_set_hash', compute_feature_set_hash(features))
            mlflow.log_param('pipeline_version', PIPELINE_VERSION)
            mlflow.log_param('preprocess_version', PREPROCESS_VERSION)
            mlflow.log_param('n_features_used', len(features))

            mlflow.log_metric('r2', out['r2'])
            mlflow.log_metric('mean_fold_r2', out['mean_fold_r2'])
            mlflow.log_metric('min_fold_r2', out['min_fold_r2'])
            mlflow.log_metric('holdout_r2', out['holdout_r2'])
            mlflow.log_metric('rmse', out['rmse'])
            mlflow.log_metric('mae', out['mae'])
            mlflow.log_metric('dummy_r2', out['dummy_r2'])
            mlflow.log_metric('dummy_holdout_r2', out['dummy_holdout_r2'])
            mlflow.log_metric('delta_r2_vs_dummy', out['delta_r2_vs_dummy'])
            if not pd.isna(out['delta_holdout_r2_vs_dummy']):
                mlflow.log_metric('delta_holdout_r2_vs_dummy', out['delta_holdout_r2_vs_dummy'])
            mlflow.log_metric('cv_time_sec', dt)

            selection_score = compute_selection_score(
                overall_r2=out['r2'],
                holdout_r2=out['holdout_r2'],
                min_fold_r2=out['min_fold_r2']
            )

            rows_scout.append({
                'stage': 'scout',
                'run_name': run_name,
                'target': target,
                'model_name': model_name,
                'feature_set': feature_set_name,
                'features_used_json': json.dumps(features),
                'r2': out['r2'],
                'mean_fold_r2': out['mean_fold_r2'],
                'min_fold_r2': out['min_fold_r2'],
                'holdout_r2': out['holdout_r2'],
                'dummy_r2': out['dummy_r2'],
                'dummy_mean_fold_r2': out['dummy_mean_fold_r2'],
                'dummy_min_fold_r2': out['dummy_min_fold_r2'],
                'dummy_holdout_r2': out['dummy_holdout_r2'],
                'delta_r2_vs_dummy': out['delta_r2_vs_dummy'],
                'delta_min_fold_r2_vs_dummy': out['delta_min_fold_r2_vs_dummy'],
                'delta_holdout_r2_vs_dummy': out['delta_holdout_r2_vs_dummy'],
                'selection_score': selection_score,
                'rmse': out['rmse'],
                'mae': out['mae'],
                'cv_time_sec': dt,
            })

        print(out['fold_df'])
        print(
            f"OOF R2={out['r2']:.4f} vs dummy={out['dummy_r2']:.4f} (delta={out['delta_r2_vs_dummy']:.4f}) | "
            f"Holdout R2={out['holdout_r2']:.4f} vs dummy={out['dummy_holdout_r2']:.4f} | "
            f"selection_score={selection_score:.4f} | "
            f"min_fold_r2={out['min_fold_r2']:.4f} vs dummy={out['dummy_min_fold_r2']:.4f}"
        )

scout_df = pd.DataFrame(rows_scout).sort_values(
    ['target', 'selection_score', 'r2', 'min_fold_r2'],
    ascending=[True, False, False, False]
).reset_index(drop=True)

print()
print('Scout results:')
display(scout_df)



Scout scope: ALL spatial groups

--- SCOUT__RF_n600_raw__C__TotalAlkalinity ---
   fold     n        r2       rmse        mae
0     1  1862 -0.214643  81.730335  63.623316
1     2  1855 -0.117466  80.715082  62.321277
2     3  1844 -0.588027  72.422624  56.196144
3     4  1917 -0.867509  79.234242  67.740310
4     5  1841 -0.653637  74.295982  58.234950
OOF R2=-0.0848 vs dummy=-0.1604 (delta=0.0756) | Holdout R2=0.1810 vs dummy=-0.0857 | selection_score=-0.1357 | min_fold_r2=-0.8675 vs dummy=-1.3037

--- SCOUT__RF_n600_Log__C__TotalAlkalinity ---
   fold     n        r2        rmse        mae
0     1  1862 -0.710908   97.000133  75.498126
1     2  1855 -0.293684   86.846348  66.945476
2     3  1844 -0.591398   72.499466  56.697522
3     4  1917 -0.455923   69.960162  57.997296
4     5  1841 -2.012832  100.284248  83.710697
OOF R2=-0.3292 vs dummy=-0.1604 (delta=-0.1689) | Holdout R2=0.0766 vs dummy=-0.0857 | selection_score=-0.4803 | min_fold_r2=-2.0128 vs dummy=-1.3037

--- SCOUT__RF_

,stage,run_name,target,model_name,feature_set,features_used_json,r2,mean_fold_r2,min_fold_r2,holdout_r2,...,dummy_mean_fold_r2,dummy_min_fold_r2,dummy_holdout_r2,delta_r2_vs_dummy,delta_min_fold_r2_vs_dummy,delta_holdout_r2_vs_dummy,selection_score,rmse,mae,cv_time_sec
0,scout,SCOUT__RF_n600_Log__C__DissolvedReactivePhosph...,Dissolved Reactive Phosphorus,RF_n600_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.094195,-0.204447,-0.656414,-0.199442,...,-0.275420,-0.858426,-0.014441,0.118773,0.202012,-0.185001,-0.204840,53.324349,32.280696,14.346722
1,scout,SCOUT__RF_n600_raw__C__DissolvedReactivePhosph...,Dissolved Reactive Phosphorus,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.067897,-0.347817,-0.815094,-0.868261,...,-0.275420,-0.858426,-0.014441,0.145072,0.043332,-0.853820,-0.380068,52.679641,37.089449,14.241693
2,scout,SCOUT__RF_n600_raw__C__ElectricalConductance,Electrical Conductance,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.008935,-0.194391,-0.612216,-0.002008,...,-0.299134,-0.756982,-0.091150,0.137520,0.144767,0.089142,-0.097696,343.443601,263.838338,14.530117
3,scout,SCOUT__RF_n600_Log__C__ElectricalConductance,Electrical Conductance,RF_n600_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.143655,-0.303106,-0.514784,0.059062,...,-0.299134,-0.756982,-0.091150,0.002801,0.242198,0.150211,-0.148645,365.654702,272.721617,14.478534
4,scout,SCOUT__RF_n600_raw__C__TotalAlkalinity,Total Alkalinity,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.084777,-0.488256,-0.867509,0.181041,...,-0.564499,-1.303729,-0.085724,0.075580,0.436220,0.266765,-0.135732,77.790114,61.676895,15.228129
5,scout,SCOUT__RF_n600_Log__C__TotalAlkalinity,Total Alkalinity,RF_n600_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.329237,-0.812949,-2.012832,0.076606,...,-0.564499,-1.303729,-0.085724,-0.168880,-0.709103,0.162330,-0.480316,86.110374,68.097844,14.196304


## Stage 2: Full finalists

For each target, we take the top 2 scout candidates and run full grouped CV.
We also save inference artifacts for those finalists.

In [11]:
def build_finalist_shortlist(scout_df_local, top_n=2):
    # Keep a union of top candidates by complementary views so we do not
    # discard globally stronger or more robust models too early.
    pieces = []

    for target_name, tdf in scout_df_local.groupby('target'):
        by_selection = tdf.sort_values(
            ['selection_score', 'r2', 'min_fold_r2'],
            ascending=[False, False, False]
        ).head(top_n)

        by_global = tdf.sort_values(
            ['r2', 'min_fold_r2', 'selection_score'],
            ascending=[False, False, False]
        ).head(top_n)

        by_robust = tdf.sort_values(
            ['min_fold_r2', 'r2', 'selection_score'],
            ascending=[False, False, False]
        ).head(1)

        shortlist = pd.concat([by_selection, by_global, by_robust], axis=0)
        shortlist = shortlist.drop_duplicates(subset=['run_name']).reset_index(drop=True)
        pieces.append(shortlist)

    return pd.concat(pieces, axis=0).reset_index(drop=True)


finalist_df = build_finalist_shortlist(scout_df, top_n=2)

print('Finalists (union shortlist):')
display(finalist_df[[
    'target',
    'feature_set',
    'model_name',
    'selection_score',
    'holdout_r2',
    'r2',
    'min_fold_r2',
    'delta_r2_vs_dummy',
    'delta_holdout_r2_vs_dummy',
    'run_name'
]])

rows_full = []
OOF_PREDS = {}

for _, row in finalist_df.iterrows():
    target = row['target']
    model_name = row['model_name']
    feature_set_name = row['feature_set']
    features = json.loads(row['features_used_json'])
    estimator = clone(MODEL_BANK[model_name])

    run_name = f'FULL__{model_name}__{feature_set_name}__{target_key(target)}'
    print()
    print(f'=== {run_name} ===')

    with mlflow.start_run(run_name=run_name):
        t0 = time.time()

        out = grouped_oof_eval(
            df_local=df,
            target=target,
            estimator=estimator,
            features_used=features,
            allowed_regions=None
        )

        dt = time.time() - t0

        preproc_path, model_path = fit_full_and_save(
            df_local=df,
            target=target,
            estimator=estimator,
            features_used=features,
            run_name=run_name
        )

        mlflow.log_param('stage', 'full')
        mlflow.log_param('target', target)
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('feature_set_name', feature_set_name)
        mlflow.log_param('split_strategy', SPLIT_STRATEGY)
        mlflow.log_param('group_definition_version', GROUP_DEFINITION_VERSION)
        mlflow.log_param('group_values_hash', compute_group_values_hash(df['spatial_group'].astype(str)))
        mlflow.log_param('feature_set_hash', compute_feature_set_hash(features))
        mlflow.log_param('pipeline_version', PIPELINE_VERSION)
        mlflow.log_param('preprocess_version', PREPROCESS_VERSION)
        mlflow.log_param('n_features_used', len(features))

        mlflow.log_metric('r2', out['r2'])
        mlflow.log_metric('mean_fold_r2', out['mean_fold_r2'])
        mlflow.log_metric('min_fold_r2', out['min_fold_r2'])
        mlflow.log_metric('holdout_r2', out['holdout_r2'])
        mlflow.log_metric('rmse', out['rmse'])
        mlflow.log_metric('mae', out['mae'])
        mlflow.log_metric('dummy_r2', out['dummy_r2'])
        mlflow.log_metric('dummy_holdout_r2', out['dummy_holdout_r2'])
        mlflow.log_metric('delta_r2_vs_dummy', out['delta_r2_vs_dummy'])
        if not pd.isna(out['delta_holdout_r2_vs_dummy']):
            mlflow.log_metric('delta_holdout_r2_vs_dummy', out['delta_holdout_r2_vs_dummy'])
        mlflow.log_metric('cv_time_sec', dt)

        mlflow.log_artifact(preproc_path, artifact_path='submission_assets')
        mlflow.log_artifact(model_path, artifact_path='submission_assets')

        OOF_PREDS[run_name] = out['pred']

        selection_score = compute_selection_score(
            overall_r2=out['r2'],
            holdout_r2=out['holdout_r2'],
            min_fold_r2=out['min_fold_r2']
        )

        rows_full.append({
            'stage': 'full',
            'run_name': run_name,
            'target': target,
            'model_name': model_name,
            'feature_set': feature_set_name,
            'features_used_json': json.dumps(features),
            'r2': out['r2'],
            'mean_fold_r2': out['mean_fold_r2'],
            'min_fold_r2': out['min_fold_r2'],
            'holdout_r2': out['holdout_r2'],
            'dummy_r2': out['dummy_r2'],
            'dummy_mean_fold_r2': out['dummy_mean_fold_r2'],
            'dummy_min_fold_r2': out['dummy_min_fold_r2'],
            'dummy_holdout_r2': out['dummy_holdout_r2'],
            'delta_r2_vs_dummy': out['delta_r2_vs_dummy'],
            'delta_min_fold_r2_vs_dummy': out['delta_min_fold_r2_vs_dummy'],
            'delta_holdout_r2_vs_dummy': out['delta_holdout_r2_vs_dummy'],
            'selection_score': selection_score,
            'rmse': out['rmse'],
            'mae': out['mae'],
            'cv_time_sec': dt,
            'preproc_path': preproc_path,
            'model_path': model_path,
        })

    print(out['fold_df'])
    print(
        f"FULL OOF R2={out['r2']:.4f} vs dummy={out['dummy_r2']:.4f} (delta={out['delta_r2_vs_dummy']:.4f}) | "
        f"Holdout R2={out['holdout_r2']:.4f} vs dummy={out['dummy_holdout_r2']:.4f} | "
        f"selection_score={selection_score:.4f} | "
        f"min_fold_r2={out['min_fold_r2']:.4f} vs dummy={out['dummy_min_fold_r2']:.4f}"
    )

full_df = pd.DataFrame(rows_full).sort_values(
    ['target', 'selection_score', 'r2', 'min_fold_r2'],
    ascending=[True, False, False, False]
).reset_index(drop=True)

print()
print('Full results:')
display(full_df)



Finalists (union shortlist):


,target,feature_set,model_name,selection_score,holdout_r2,r2,min_fold_r2,delta_r2_vs_dummy,delta_holdout_r2_vs_dummy,run_name
0,Dissolved Reactive Phosphorus,C,RF_n600_Log,-0.204840,-0.199442,-0.094195,-0.656414,0.118773,-0.185001,SCOUT__RF_n600_Log__C__DissolvedReactivePhosph...
1,Dissolved Reactive Phosphorus,C,RF_n600_raw,-0.380068,-0.868261,-0.067897,-0.815094,0.145072,-0.853820,SCOUT__RF_n600_raw__C__DissolvedReactivePhosph...
2,Electrical Conductance,C,RF_n600_raw,-0.097696,-0.002008,-0.008935,-0.612216,0.137520,0.089142,SCOUT__RF_n600_raw__C__ElectricalConductance
3,Electrical Conductance,C,RF_n600_Log,-0.148645,0.059062,-0.143655,-0.514784,0.002801,0.150211,SCOUT__RF_n600_Log__C__ElectricalConductance
4,Total Alkalinity,C,RF_n600_raw,-0.135732,0.181041,-0.084777,-0.867509,0.075580,0.266765,SCOUT__RF_n600_raw__C__TotalAlkalinity
5,Total Alkalinity,C,RF_n600_Log,-0.480316,0.076606,-0.329237,-2.012832,-0.168880,0.162330,SCOUT__RF_n600_Log__C__TotalAlkalinity



=== FULL__RF_n600_Log__C__DissolvedReactivePhosphorus ===
   fold     n        r2       rmse        mae
0     1  1862 -0.656414  82.848289  61.542996
1     2  1855 -0.067828  57.214412  38.782327
2     3  1844 -0.106525  47.487536  25.342381
3     4  1917 -0.130752  27.783952  17.004856
4     5  1841 -0.060717  33.021743  18.989611
FULL OOF R2=-0.0942 vs dummy=-0.2130 (delta=0.1188) | Holdout R2=-0.1994 vs dummy=-0.0144 | selection_score=-0.2048 | min_fold_r2=-0.6564 vs dummy=-0.8584

=== FULL__RF_n600_raw__C__DissolvedReactivePhosphorus ===
   fold     n        r2       rmse        mae
0     1  1862 -0.422190  76.767574  59.846492
1     2  1855 -0.015770  55.802351  43.115761
2     3  1844 -0.089071  47.111511  29.099126
3     4  1917 -0.815094  35.201404  26.337864
4     5  1841 -0.396958  37.895880  27.199452
FULL OOF R2=-0.0679 vs dummy=-0.2130 (delta=0.1451) | Holdout R2=-0.8683 vs dummy=-0.0144 | selection_score=-0.3801 | min_fold_r2=-0.8151 vs dummy=-0.8584

=== FULL__RF_n600_r

,stage,run_name,target,model_name,feature_set,features_used_json,r2,mean_fold_r2,min_fold_r2,holdout_r2,...,dummy_holdout_r2,delta_r2_vs_dummy,delta_min_fold_r2_vs_dummy,delta_holdout_r2_vs_dummy,selection_score,rmse,mae,cv_time_sec,preproc_path,model_path
0,full,FULL__RF_n600_Log__C__DissolvedReactivePhosphorus,Dissolved Reactive Phosphorus,RF_n600_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.094195,-0.204447,-0.656414,-0.199442,...,-0.014441,0.118773,0.202012,-0.185001,-0.204840,53.324349,32.280696,14.491037,../models/final_deadline_mvp4\FULL__RF_n600_Lo...,../models/final_deadline_mvp4\FULL__RF_n600_Lo...
1,full,FULL__RF_n600_raw__C__DissolvedReactivePhosphorus,Dissolved Reactive Phosphorus,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.067897,-0.347817,-0.815094,-0.868261,...,-0.014441,0.145072,0.043332,-0.853820,-0.380068,52.679641,37.089449,14.082798,../models/final_deadline_mvp4\FULL__RF_n600_ra...,../models/final_deadline_mvp4\FULL__RF_n600_ra...
2,full,FULL__RF_n600_raw__C__ElectricalConductance,Electrical Conductance,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.008935,-0.194391,-0.612216,-0.002008,...,-0.091150,0.137520,0.144767,0.089142,-0.097696,343.443601,263.838338,14.236601,../models/final_deadline_mvp4\FULL__RF_n600_ra...,../models/final_deadline_mvp4\FULL__RF_n600_ra...
3,full,FULL__RF_n600_Log__C__ElectricalConductance,Electrical Conductance,RF_n600_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.143655,-0.303106,-0.514784,0.059062,...,-0.091150,0.002801,0.242198,0.150211,-0.148645,365.654702,272.721617,14.499439,../models/final_deadline_mvp4\FULL__RF_n600_Lo...,../models/final_deadline_mvp4\FULL__RF_n600_Lo...
4,full,FULL__RF_n600_raw__C__TotalAlkalinity,Total Alkalinity,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.084777,-0.488256,-0.867509,0.181041,...,-0.085724,0.075580,0.436220,0.266765,-0.135732,77.790114,61.676895,14.016945,../models/final_deadline_mvp4\FULL__RF_n600_ra...,../models/final_deadline_mvp4\FULL__RF_n600_ra...
5,full,FULL__RF_n600_Log__C__TotalAlkalinity,Total Alkalinity,RF_n600_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.329237,-0.812949,-2.012832,0.076606,...,-0.085724,-0.168880,-0.709103,0.162330,-0.480316,86.110374,68.097844,14.242463,../models/final_deadline_mvp4\FULL__RF_n600_Lo...,../models/final_deadline_mvp4\FULL__RF_n600_Lo...


## Freeze Manifest A

This chooses one safe anchor model per target from the full finalists.
For DRP, we prefer the safer linear models.

In [ ]:
TARGET_GLOBAL_R2_FLOOR = {
    'Total Alkalinity': 0.00,
    'Electrical Conductance': 0.00,
    'Dissolved Reactive Phosphorus': -0.20,
}

# RF-only mode
DRP_SAFE_MODELS = [
    'RF_n600_raw',
    'RF_n600_Log',
]



In [ ]:
manifest_A = {}
manifest_B = {}


def pick_with_gate(tdf, preferred_model=None):
    # Hard gate: prefer rows that beat dummy on pseudo-holdout.
    gated = tdf[tdf['delta_holdout_r2_vs_dummy'] > 0].copy()
    pool = gated if not gated.empty else tdf.copy()

    if preferred_model is not None:
        pref = pool[pool['model_name'] == preferred_model].copy()
        if not pref.empty:
            pool = pref

    pool = pool.sort_values(
        ['delta_holdout_r2_vs_dummy', 'holdout_r2', 'selection_score', 'r2'],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)
    return pool.iloc[0]


for target in TARGET_COLS:
    tdf = full_df[full_df['target'] == target].copy().reset_index(drop=True)

    if target == 'Total Alkalinity':
        chosen_A = pick_with_gate(tdf, preferred_model='RF_n600_raw')
        chosen_B = chosen_A

    elif target == 'Electrical Conductance':
        chosen_A = pick_with_gate(tdf, preferred_model='RF_n600_Log')
        chosen_B = pick_with_gate(tdf, preferred_model=None)

    elif target == 'Dissolved Reactive Phosphorus':
        chosen_A = pick_with_gate(tdf, preferred_model='RF_n600_Log')
        chosen_B = chosen_A

    manifest_A[target] = chosen_A.to_dict()
    manifest_B[target] = chosen_B.to_dict()


DRP_DELTA_HOLDOUT = float(manifest_A['Dissolved Reactive Phosphorus']['delta_holdout_r2_vs_dummy'])
DRP_USE_MODEL = DRP_DELTA_HOLDOUT > 0.0

print('DRP_USE_MODEL:', DRP_USE_MODEL, '| delta_holdout_vs_dummy:', round(DRP_DELTA_HOLDOUT, 6))

print('=== MANIFEST A (GATED) ===')
print(json.dumps({
    k: {
        'run_name': v['run_name'],
        'model_name': v['model_name'],
        'holdout_r2': float(v['holdout_r2']) if pd.notna(v['holdout_r2']) else None,
        'dummy_holdout_r2': float(v['dummy_holdout_r2']) if pd.notna(v['dummy_holdout_r2']) else None,
        'delta_holdout_r2_vs_dummy': float(v['delta_holdout_r2_vs_dummy']) if pd.notna(v['delta_holdout_r2_vs_dummy']) else None,
        'selection_score': float(v['selection_score']),
        'r2': float(v['r2']),
        'min_fold_r2': float(v['min_fold_r2']),
    }
    for k, v in manifest_A.items()
}, indent=2))

print('\n=== MANIFEST B (GATED) ===')
print(json.dumps({
    k: {
        'run_name': v['run_name'],
        'model_name': v['model_name'],
        'holdout_r2': float(v['holdout_r2']) if pd.notna(v['holdout_r2']) else None,
        'dummy_holdout_r2': float(v['dummy_holdout_r2']) if pd.notna(v['dummy_holdout_r2']) else None,
        'delta_holdout_r2_vs_dummy': float(v['delta_holdout_r2_vs_dummy']) if pd.notna(v['delta_holdout_r2_vs_dummy']) else None,
        'selection_score': float(v['selection_score']),
        'r2': float(v['r2']),
        'min_fold_r2': float(v['min_fold_r2']),
    }
    for k, v in manifest_B.items()
}, indent=2))



## Diagnostic For Retraining Before Making Submission

In [ ]:
cols = ['swir22', 'NDMI', 'MNDWI', 'pet'] + TARGET_COLS
display(df[cols].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]).T)

## Submission helper

This loads the saved artifacts for a chosen manifest entry and generates clipped predictions.

In [ ]:
def predict_from_manifest_entry(entry, df_val_local):
    '''
    Predict from a frozen manifest entry using the saved
    preprocessor and model artifacts.
    '''
    feats = json.loads(entry['features_used_json'])
    pre = joblib.load(entry['preproc_path'])
    mdl = joblib.load(entry['model_path'])

    X = df_val_local[feats]
    pred = mdl.predict(pre.transform(X))
    pred = np.asarray(pred, dtype=float)

    return np.clip(pred, 0, None)

## Build Shot A / B / C

- **Shot A** = safe anchor from Manifest A
- **Shot B** = EC aggressive + DRP safe shrink
- **Shot C** = hedge blend between A and B

In [ ]:
df_val = pd.read_parquet('../data/interim/water_quality_mvp_validation.parquet')
df_val = engineer_features(df_val)

tpl = pd.read_csv('../data/raw/submission_template.csv')
tpl = make_row_id_template(tpl)

# Validate that all needed features exist in validation
needed_feats = set()
for t in ['Total Alkalinity', 'Electrical Conductance']:
    needed_feats.update(json.loads(manifest_A[t]['features_used_json']))
    needed_feats.update(json.loads(manifest_B[t]['features_used_json']))

if DRP_USE_MODEL:
    needed_feats.update(json.loads(manifest_A['Dissolved Reactive Phosphorus']['features_used_json']))

missing_feats = sorted([f for f in needed_feats if f not in df_val.columns])
if missing_feats:
    raise RuntimeError(f'Missing validation features: {missing_feats}')


def clip_by_train_quantile(pred, target, q_hi=0.995):
    hi = float(df[target].quantile(q_hi))
    return np.clip(np.asarray(pred, dtype=float), 0, hi)


drp_train_median = float(df['Dissolved Reactive Phosphorus'].median())

# -------------------------
# Shot A: safe anchor
# -------------------------
shotA = tpl.copy()

pred_ta_a = predict_from_manifest_entry(manifest_A['Total Alkalinity'], df_val)
pred_ec_a = predict_from_manifest_entry(manifest_A['Electrical Conductance'], df_val)

shotA['Total Alkalinity'] = clip_by_train_quantile(pred_ta_a, 'Total Alkalinity')
shotA['Electrical Conductance'] = clip_by_train_quantile(pred_ec_a, 'Electrical Conductance')

if DRP_USE_MODEL:
    drp_a = predict_from_manifest_entry(manifest_A['Dissolved Reactive Phosphorus'], df_val)
    shotA['Dissolved Reactive Phosphorus'] = clip_by_train_quantile(drp_a, 'Dissolved Reactive Phosphorus')
    print('Shot A DRP model:', manifest_A['Dissolved Reactive Phosphorus']['run_name'])
else:
    shotA['Dissolved Reactive Phosphorus'] = drp_train_median
    print('Shot A DRP fallback: train median')

assert_submission_integrity(shotA, tpl, TARGET_COLS)

# -------------------------
# Shot B: modest challenger
# -------------------------
shotB = tpl.copy()
shotB['Total Alkalinity'] = shotA['Total Alkalinity']

if manifest_B['Electrical Conductance']['run_name'] != manifest_A['Electrical Conductance']['run_name']:
    ec_safe = shotA['Electrical Conductance'].values
    ec_chal = predict_from_manifest_entry(manifest_B['Electrical Conductance'], df_val)
    ec_blend = 0.35 * ec_safe + 0.65 * ec_chal
    shotB['Electrical Conductance'] = clip_by_train_quantile(ec_blend, 'Electrical Conductance')
    print('Shot B EC challenger blend:', manifest_B['Electrical Conductance']['run_name'])
else:
    shotB['Electrical Conductance'] = shotA['Electrical Conductance']
    print('Shot B EC kept from Manifest A')

if DRP_USE_MODEL:
    drp_model = predict_from_manifest_entry(manifest_A['Dissolved Reactive Phosphorus'], df_val)
    drp_blend = 0.20 * drp_model + 0.80 * drp_train_median
    shotB['Dissolved Reactive Phosphorus'] = clip_by_train_quantile(drp_blend, 'Dissolved Reactive Phosphorus')
    print('Shot B DRP model+median alpha=0.20')
else:
    shotB['Dissolved Reactive Phosphorus'] = drp_train_median
    print('Shot B DRP fallback: train median')

assert_submission_integrity(shotB, tpl, TARGET_COLS)

# -------------------------
# Shot C: stronger DRP hedge
# -------------------------
shotC = tpl.copy()
shotC['Total Alkalinity'] = clip_by_train_quantile(
    0.85 * shotA['Total Alkalinity'] + 0.15 * shotB['Total Alkalinity'],
    'Total Alkalinity'
)
shotC['Electrical Conductance'] = clip_by_train_quantile(
    0.50 * shotA['Electrical Conductance'] + 0.50 * shotB['Electrical Conductance'],
    'Electrical Conductance'
)

if DRP_USE_MODEL:
    drp_model = predict_from_manifest_entry(manifest_A['Dissolved Reactive Phosphorus'], df_val)
    drp_blend_c = 0.10 * drp_model + 0.90 * drp_train_median
    shotC['Dissolved Reactive Phosphorus'] = clip_by_train_quantile(drp_blend_c, 'Dissolved Reactive Phosphorus')
    print('Shot C DRP model+median alpha=0.10')
else:
    shotC['Dissolved Reactive Phosphorus'] = drp_train_median
    print('Shot C DRP fallback: train median')

assert_submission_integrity(shotC, tpl, TARGET_COLS)

# -------------------------
# Save
# -------------------------
stamp = datetime.now().strftime('%Y%m%d_%H%M')
pathA = f'../data/submission/submission_{stamp}_A_safe.csv'
pathB = f'../data/submission/submission_{stamp}_B_challenger_blend.csv'
pathC = f'../data/submission/submission_{stamp}_C_hedge.csv'

os.makedirs('../data/submission', exist_ok=True)

shotA.drop(columns=['row_id']).to_csv(pathA, index=False)
shotB.drop(columns=['row_id']).to_csv(pathB, index=False)
shotC.drop(columns=['row_id']).to_csv(pathC, index=False)

print('Saved files:')
print('A:', pathA)
print('B:', pathB)
print('C:', pathC)



## Submission diagnostics

This final cell prints simple distribution summaries for the three submission variants.

In [ ]:
def summarize_shot(shot_df, name):
    print(f'\n{name} stats')
    stats = shot_df[TARGET_COLS].describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    ).T
    display(stats[['min', '1%', '5%', '50%', 'mean', '95%', '99%', 'max']])

summarize_shot(shotA, 'Shot A')
summarize_shot(shotB, 'Shot B')
summarize_shot(shotC, 'Shot C')

print('\nMean absolute deltas vs Shot A')
delta_tbl = pd.DataFrame({
    'target': TARGET_COLS,
    'B_vs_A_mae': [float(np.mean(np.abs(shotB[t] - shotA[t]))) for t in TARGET_COLS],
    'C_vs_A_mae': [float(np.mean(np.abs(shotC[t] - shotA[t]))) for t in TARGET_COLS],
})
display(delta_tbl)

tracker = pd.DataFrame([
    {'file': pathA, 'hypothesis': 'Safety anchor (lowest variance)'},
    {'file': pathC, 'hypothesis': 'Balanced hedge between A and B'},
    {'file': pathB, 'hypothesis': 'Most aggressive on EC/DRP challenger blend'},
])

print('\nSubmission tracker:')
display(tracker)

print('\nSuggested upload order: A -> C -> B')